In [1]:
import pandas as pd

df = pd.read_csv("music_metadata_not_classification.csv") 
df

,source,song_name,song_by,Classification,Mood,Video Theme,Instrument,Genre,BPM,LMM Description,Number,Unnamed: 11
0,Artlist,My Rhapsody Sounds - Short Version A,Assaf Ayalon,Happy,"Uplifting, Happy, Carefree, Love, Playful","Business, Food, Education, Lifestyle, Urban","Acoustic Guitar, Keys","Cinematic, Acoustic, Pop, Folk, Children, Corp...",145.0,A positive and uplifting acoustic folk track w...,1,NaN
1,Artlist,Laid Back - Short Version A,The Mind Sweepers,Agitated,"Powerful, Serious, Angry","Road Trip, Sport & Fitness, Fashion, Industry","Electric, Guitar, Acoustic Drums",Rock,78.0,This is a powerful and energetic rock music tr...,2,NaN
2,Artlist,Far Taj,ZISO,Vigorous,"Uplifting, Powerful, Carefree, Groovy","Travel, Shorts","Ethnic, Electronic Drums, Bass","World, Electronic, Hip Hop",96.0,A traditional Indian Bhangra track with modern...,3,NaN
3,Artlist,The Stones - Short Version,Wild Tulip,Sad,"Love, Serious, Dramatic, Sad, Hopeful","Time-Lapse, Documentary, Road Trip, Medical, L...",Piano,Cinematic,69.0,This piece is a solo piano instrumental with a...,4,NaN
4,Artlist,Fixed - Short Version B,Swirling Ship,Serious,"Serious, Dramatic, Scary, Dark","Time-Lapse, Drone Shots, Nature, Slow Motion","Electric Guitar, Synth, Electronic Drums, Pads","Ambient, Country, Cinematic",121.0,"The music is mysterious and dramatic, featurin...",5,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
195,envato,Orchestral News Intro,Tomasz_Redman,Dreamy,"announcement, background, broadcast news, broa...",NaN,NaN,NaN,125.0,This is a dynamic and uplifting music track th...,196,NaN
196,envato,Upbeat Happy Fun Logo,puremusic,Happy,"acoustic, bouncy, bright, catchy, cheerful, ch...",NaN,NaN,NaN,NaN,"A positive, upbeat, cheerful, and happy acoust...",197,NaN
197,envato,Happy Birthday In Paris,Music-Ideas,Dreamy,"accordion, ads, advertising, bigband, birthday...",NaN,NaN,NaN,120.0,A fun and lively Latin track featuring a varie...,198,NaN
198,envato,Funny Game Loop,honey_lemon,Humorous,"acoustic, bells, cartoon, children, comedy, co...",NaN,NaN,NaN,170.0,"A casual, jazzy, swing music with vibraphone, ...",199,wav


In [2]:
# for few-shot learing, use Artlist.io's data as examples
few_shot_json = []
# delete duplicate for examples
used_pairs = set()

category_map = {
    "Mood": "Mood",
    "Video Theme": "Video Theme",
    "Instrument": "Instrument",
    "Genre": "Genre"
}

# 100 rows (Artilist.io)
df_subset = df.head(100)

for _, row in df_subset.iterrows():
    for col in category_map:
        if pd.notna(row[col]):
            tags = [tag.strip() for tag in row[col].split(',')]
            for tag in tags:
                pair = (tag, category_map[col])
                if pair not in used_pairs:
                    few_shot_json.append({
                        "label": tag,
                        "category": category_map[col]
                    })
                    used_pairs.add(pair)

print("Total unique (label, category) pairs:", len(used_pairs))                    
# this is for column first
# for col in ["Mood", "Video Theme", "Instrument", "Genre"]:
#     for row in df_subset[col].dropna():
#         tags = [tag.strip() for tag in row.split(',')]
#         for tag in tags:
#             few_shot_json.append({
#                 "label": tag,
#                 "category": category_map[col]
#             })

Total unique (label, category) pairs: 92


In [3]:
# VER the method 
import json
print(json.dumps(few_shot_json[:10], ensure_ascii=False, indent=2))

[
  {
    "label": "Uplifting",
    "category": "Mood"
  },
  {
    "label": "Happy",
    "category": "Mood"
  },
  {
    "label": "Carefree",
    "category": "Mood"
  },
  {
    "label": "Love",
    "category": "Mood"
  },
  {
    "label": "Playful",
    "category": "Mood"
  },
  {
    "label": "Business",
    "category": "Video Theme"
  },
  {
    "label": "Food",
    "category": "Video Theme"
  },
  {
    "label": "Education",
    "category": "Video Theme"
  },
  {
    "label": "Lifestyle",
    "category": "Video Theme"
  },
  {
    "label": "Urban",
    "category": "Video Theme"
  }
]


In [4]:
# for prompt using
few_shot_prompt_lines = [
    f"{item['label']} → {item['category']}" for item in few_shot_json
]

few_shot_prompt = "\n".join(few_shot_prompt_lines)
print(few_shot_prompt)

Uplifting → Mood
Happy → Mood
Carefree → Mood
Love → Mood
Playful → Mood
Business → Video Theme
Food → Video Theme
Education → Video Theme
Lifestyle → Video Theme
Urban → Video Theme
Acoustic Guitar → Instrument
Keys → Instrument
Cinematic → Genre
Acoustic → Genre
Pop → Genre
Folk → Genre
Children → Genre
Corporate → Genre
Powerful → Mood
Serious → Mood
Angry → Mood
Road Trip → Video Theme
Sport & Fitness → Video Theme
Fashion → Video Theme
Industry → Video Theme
Electric → Instrument
Guitar → Instrument
Acoustic Drums → Instrument
Rock → Genre
Groovy → Mood
Travel → Video Theme
Shorts → Video Theme
Ethnic → Instrument
Electronic Drums → Instrument
Bass → Instrument
World → Genre
Electronic → Genre
Hip Hop → Genre
Dramatic → Mood
Sad → Mood
Hopeful → Mood
Time-Lapse → Video Theme
Documentary → Video Theme
Medical → Video Theme
Landscape → Video Theme
Nature → Video Theme
Slow Motion → Video Theme
Piano → Instrument
Scary → Mood
Dark → Mood
Drone Shots → Video Theme
Electric Guitar → In

In [5]:
# range for envato
target_rows = df.loc[100:199, 'Mood'].dropna()

# all tag to avoid duplicate
all_tags = set()
row_to_tags = {}

for idx, row in target_rows.items():
    tags = [tag.strip() for tag in row.split(',') if tag.strip()]
    row_to_tags[idx] = tags
    all_tags.update(tags)

In [6]:
# because one coversation cannot include too much tags, so split to chunks
def chunk_list(lst, chunk_size):
    for i in range(0, len(lst), chunk_size):
        yield lst[i:i + chunk_size]
        
tag_chunks = list(chunk_list(sorted(all_tags), chunk_size = 150))
print(f"Total {len(tag_chunks)} chunks")

Total 7 chunks


In [7]:
# test for API connection
from dotenv import load_dotenv
load_dotenv()
import os
from openai import OpenAI
client = OpenAI()
try:
    response = client.chat.completions.create(
        model="gpt-4",
        messages=[{"role": "user", "content": "hello"}],
        timeout=10
    )
    print("✅ 成功連線！", response.choices[0].message.content)
except Exception as e:
    print("❌ 發生錯誤：", e)

✅ 成功連線！ Hello! How can I assist you today?


In [8]:
tag_to_category = {}

for chunk in tag_chunks:
    tag_list_text = "\n".join(chunk)

    # full prompt
    # 不合適的可以多一個 OTHERS 的分類 (4+1)
    # A 的 DIMENSION 如果是空的也沒關係
    classification_prompt = f"""
    您是一位音樂領域的分類標籤專家。請你負責分類音樂標籤到以下四個種類。

    1. Mood: 有關於情緒
    2. Video Theme: 有關於影片主題
    3. Instrument: 有關於樂器
    4. Genre: 有關於音樂流派

    如果都不屬於，請判斷其是否對做音樂推薦有益，如果是請回答 others, 否則回答 useless。

    以下是一些音樂標籤與對應分類：

    {few_shot_prompt}

    請將以下標籤分類為：「Mood」、「Video Theme」、「Instrument」、「Genre」、「Others」或「Useless」：
    {tag_list_text}

    請依照以下格式輸出：
    標籤 → 類別
    """

    # call GPT
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "user", "content": classification_prompt}
        ],
        temperature=0
    )
    
    # parse mapping: tag → category
    response_text = response.choices[0].message.content
    result_lines = response_text.split('\n')

    for line in result_lines:
        if "→" in line:
            tag, cat = [s.strip() for s in line.split("→")]
            tag_to_category[tag] = cat

In [9]:
# VAL all tag are definitely classified
print(len(all_tags))
print(len(tag_to_category))

915
915


In [10]:
mood_labels, video_theme_labels, instrument_labels, genre_labels, others_labels, useless_labels = [], [], [], [], [], []

for idx in row_to_tags:
    mood, video_theme, instrument, genre, others, useless = [], [], [], [], [], []

    for tag in row_to_tags[idx]:
        category = tag_to_category.get(tag, "others").lower()

        if category == "mood":
            mood.append(tag)
        elif category == "video theme":
            video_theme.append(tag)
        elif category == "instrument":
            instrument.append(tag)
        elif category == "genre":
            genre.append(tag)
        elif category == "useless":
            useless.append(tag)
        else:
            others.append(tag)

    mood_labels.append(", ".join(mood))
    video_theme_labels.append(", ".join(video_theme))
    instrument_labels.append(", ".join(instrument))
    genre_labels.append(", ".join(genre))
    others_labels.append(", ".join(others))
    useless_labels.append(", ".join(useless))

df.loc[row_to_tags.keys(), "Mood_Label"] = mood_labels
df.loc[row_to_tags.keys(), "Video_Theme_Label"] = video_theme_labels
df.loc[row_to_tags.keys(), "Instrument_Label"] = instrument_labels
df.loc[row_to_tags.keys(), "Genre_Label"] = genre_labels
df.loc[row_to_tags.keys(), "Others"] = others_labels
df.loc[row_to_tags.keys(), "Useless"] = useless_labels

In [11]:
df

,source,song_name,song_by,Classification,Mood,Video Theme,Instrument,Genre,BPM,LMM Description,Number,Unnamed: 11,Mood_Label,Video_Theme_Label,Instrument_Label,Genre_Label,Others,Useless
0,Artlist,My Rhapsody Sounds - Short Version A,Assaf Ayalon,Happy,"Uplifting, Happy, Carefree, Love, Playful","Business, Food, Education, Lifestyle, Urban","Acoustic Guitar, Keys","Cinematic, Acoustic, Pop, Folk, Children, Corp...",145.0,A positive and uplifting acoustic folk track w...,1,NaN,nan,nan,nan,nan,nan,nan
1,Artlist,Laid Back - Short Version A,The Mind Sweepers,Agitated,"Powerful, Serious, Angry","Road Trip, Sport & Fitness, Fashion, Industry","Electric, Guitar, Acoustic Drums",Rock,78.0,This is a powerful and energetic rock music tr...,2,NaN,nan,nan,nan,nan,nan,nan
2,Artlist,Far Taj,ZISO,Vigorous,"Uplifting, Powerful, Carefree, Groovy","Travel, Shorts","Ethnic, Electronic Drums, Bass","World, Electronic, Hip Hop",96.0,A traditional Indian Bhangra track with modern...,3,NaN,nan,nan,nan,nan,nan,nan
3,Artlist,The Stones - Short Version,Wild Tulip,Sad,"Love, Serious, Dramatic, Sad, Hopeful","Time-Lapse, Documentary, Road Trip, Medical, L...",Piano,Cinematic,69.0,This piece is a solo piano instrumental with a...,4,NaN,nan,nan,nan,nan,nan,nan
4,Artlist,Fixed - Short Version B,Swirling Ship,Serious,"Serious, Dramatic, Scary, Dark","Time-Lapse, Drone Shots, Nature, Slow Motion","Electric Guitar, Synth, Electronic Drums, Pads","Ambient, Country, Cinematic",121.0,"The music is mysterious and dramatic, featurin...",5,NaN,nan,nan,nan,nan,nan,nan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,envato,Orchestral News Intro,Tomasz_Redman,Dreamy,"announcement, background, broadcast news, broa...",NaN,NaN,NaN,125.0,This is a dynamic and uplifting music track th...,196,NaN,"energetic, epic, powerful, solemn, uplifting","announcement, broadcast news, broadcasting, bu...",strings,corporate,global,"background, company, ident, report"
196,envato,Upbeat Happy Fun Logo,puremusic,Happy,"acoustic, bouncy, bright, catchy, cheerful, ch...",NaN,NaN,NaN,NaN,"A positive, upbeat, cheerful, and happy acoust...",197,NaN,"bouncy, bright, catchy, cheerful, energetic, f...","commercial, happy logo, intro, kids, logo, sum...","claps, ukulele","acoustic, children","melody, youth",ident
197,envato,Happy Birthday In Paris,Music-Ideas,Dreamy,"accordion, ads, advertising, bigband, birthday...",NaN,NaN,NaN,120.0,A fun and lively Latin track featuring a varie...,198,NaN,"cheerful, funny, happy, lively, playful, upbeat","ads, advertising, birthday, broadway, casino, ...","accordion, piano, trumpets","bigband, jazz, retro","france, french, paris",Emily
198,envato,Funny Game Loop,honey_lemon,Humorous,"acoustic, bells, cartoon, children, comedy, co...",NaN,NaN,NaN,170.0,"A casual, jazzy, swing music with vibraphone, ...",199,wav,"comical, fun, funny, laugh, smile, soft","cartoon, comedy, comic, kids, short, summer, tv",bells,"acoustic, children, folk, jazz",loop,"how, stand up"


In [12]:
columns_to_replace = {
    "Mood": "Mood_Label",
    "Video Theme": "Video_Theme_Label",
    "Instrument": "Instrument_Label",
    "Genre": "Genre_Label"
}

for target_col, label_col in columns_to_replace.items():
    df.loc[df['source'] == 'envato', target_col] = df.loc[df['source'] == 'envato', label_col]

In [13]:
df.drop(columns=[
    "Mood_Label", 
    "Video_Theme_Label", 
    "Instrument_Label", 
    "Genre_Label", 
    "Number",
    "Unnamed: 11"
], inplace=True, errors='ignore')

In [14]:
df.to_csv("final_dataset.csv", index=False)